    NOTE : In this notebook, we will be using cov_type as HC3 to get accurate CI's and p-values, as proved in
    4.LinearRegression.ipynb (its residual-vs-fitted diagnostics show heteroskedastic errors). The response
    variable is log(ROI) — the near-normal scale validated in 3.DistributionFitting.ipynb and adopted as the
    primary response in 4.LinearRegression.ipynb; coefficients read as percentage effects on ROI, exp(β) − 1.
    A raw-ROI refit of the same ladders closes this notebook as a labelled scale-robustness check.

    All tests are evaluated at the significance level α = 0.05 (two-sided); the 95% confidence intervals correspond to the same level.


## Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv('CleanedDataSet.csv')
df['Scale_Bin'] = pd.cut(df['Movie_Scale'], bins=range(0, 11)).cat.remove_unused_categories()
df['log_ROI'] = np.log(df['ROI'])   # primary response (per 3.DistributionFitting.ipynb / 4.LinearRegression.ipynb)

In [2]:
bools = ['Remake', 'Franchise', 'Holiday']
genres = [  'Genre_adult', 'Genre_animation', 'Genre_comedy', 'Genre_documentary',
            'Genre_drama', 'Genre_fantasy', 'Genre_horror', 'Genre_love_story',
            'Genre_masala', 'Genre_mythological', 'Genre_rom_com', 'Genre_suspense',
            'Genre_thriller']
cast_novelty = ['New_Actor', 'New_Director', 'New_Music_Director']

focus_cols = bools + genres + cast_novelty

Y = df['log_ROI']

## H1 : Festive Timing
**Does holiday slot survive a like-to-like scaled comparison**

- What does the unadjusted comparison say?

In [3]:
X = df['Holiday']
X = sm.add_constant(X)
rawmodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel.params['Holiday'], rawmodel.pvalues['Holiday']
lo, hi = rawmodel.conf_int().loc['Holiday']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}(approx)   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Holiday coefficient : +0.2786  
p-value = 0.0007(approx)   
95% CI [+0.1170, +0.4402]


- What does a content-adjusted (scale not yet controlled) model say?

In [4]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel.params['Holiday'], ROImodel.pvalues['Holiday']
lo, hi = ROImodel.conf_int().loc['Holiday']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Holiday coefficient : +0.1057  
p-value = 0.1321   
95% CI [-0.0319, +0.2434]


- What does a scale adjustment alone model say?

In [5]:
X = "log_ROI ~ Holiday" + " + C(Scale_Bin)"
raw_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale.params['Holiday[T.True]'], raw_scale.pvalues['Holiday[T.True]']
lo, hi = raw_scale.conf_int().loc['Holiday[T.True]']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")


Holiday coefficient : +0.0184  
p-value = 0.7641   
95% CI [-0.1020, +0.1389]


- what does a full model considering everything say?

In [6]:
X = "log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale.params['Holiday[T.True]'], full_scale.pvalues['Holiday[T.True]']
lo, hi = full_scale.conf_int().loc['Holiday[T.True]']
print(f"Holiday coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Holiday coefficient : -0.0153  
p-value = 0.7927   
95% CI [-0.1291, +0.0985]


### Result: **H1 is rejected** (response = log(ROI); % ROI = exp(β) − 1)

| Model | Holiday effect, log(ROI) | ≈ % ROI | 95% CI | p-value |
|---|---|---|---|---|
| Unadjusted | *+0.2786* | +32% | [+0.1170, +0.4402] | 0.0007 |
| Content-adjusted, scale not controlled | *+0.1057* | +11% | [-0.0319, +0.2434] | 0.132 |
| Scale-adjusted only | *+0.0184* | +2% | [-0.1020, +0.1389] | 0.764 |
| Scale + content (full) | *-0.0153* | -2% | [-0.1291, +0.0985] | 0.793 |


1. The unadjusted holiday premium is +0.28 log-points (≈ +32% ROI), but content adjustment alone absorbs ~62% of it, and scale adjustment absorbs essentially all of the rest.

2. In the full model the effect is −0.02 (p = 0.79): once what films are and how big they are are held fixed, festival timing has no detectable effect on returns.

3. The raw-ROI comparison at the end of this notebook tells the same story (full model +0.06, p = 0.18), so the verdict does not depend on the scale choice.

4. This is consistent with 3.DistributionFitting.ipynb (Step 8): at high scale, festival films still show a large log-revenue gap ((8,9] +0.21, shuffle p = 0.023; (9,10] +0.56, about 76% more revenue, shuffle p = 0.001). That is a revenue effect, not a returns effect - the festival premium concentrates exactly where productions are biggest, so the extra revenue does not become extra ROI. Timing buys reach, not returns.

#### *Verdict : Holiday Slot does not survive*

## H2: Remakes and Franchise
**Do the remake and franchise slots survive a like-to-like scaled comparison**

### 1. Franchise

- What does the unadjusted comparison say? (Franchise)

In [7]:
X = df['Franchise']
X = sm.add_constant(X)
rawmodel_f = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel_f.params['Franchise'], rawmodel_f.pvalues['Franchise']
lo, hi = rawmodel_f.conf_int().loc['Franchise']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +1.7019  
p-value = 0.0000   
95% CI [+1.4597, +1.9440]


- What does a content-adjusted (scale not yet controlled) model say? (Franchise)

In [8]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel_f = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel_f.params['Franchise'], ROImodel_f.pvalues['Franchise']
lo, hi = ROImodel_f.conf_int().loc['Franchise']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +1.1722  
p-value = 0.0000   
95% CI [+0.9490, +1.3954]


- What does a scale adjustment alone model say? (Franchise)

In [9]:
X = "log_ROI ~ Franchise + C(Scale_Bin)"
raw_scale_f = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale_f.params['Franchise[T.True]'], raw_scale_f.pvalues['Franchise[T.True]']
lo, hi = raw_scale_f.conf_int().loc['Franchise[T.True]']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +0.3794  
p-value = 0.0001   
95% CI [+0.1869, +0.5719]


- What does a full model considering everything say? (Franchise)

In [10]:
X = "log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale_f = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale_f.params['Franchise[T.True]'], full_scale_f.pvalues['Franchise[T.True]']
lo, hi = full_scale_f.conf_int().loc['Franchise[T.True]']
print(f"Franchise coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Franchise coefficient : +0.3356  
p-value = 0.0006   
95% CI [+0.1438, +0.5273]


### Result: **Franchise premium survives** (response = log(ROI))

| Model | Franchise effect, log(ROI) | ≈ % ROI | 95% CI | p-value |
|---|---|---|---|---|
| Unadjusted | *+1.7019* | +448% | [+1.4597, +1.9440] | < 0.0001 |
| Content-adjusted, scale not controlled | *+1.1722* | +223% | [+0.9490, +1.3954] | < 0.0001 |
| Scale-adjusted only | *+0.3794* | +46% | [+0.1869, +0.5719] | 0.0001 |
| Scale + content (full) | *+0.3356* | +40% | [+0.1438, +0.5273] | 0.0006 |


1. The unadjusted franchise premium is huge (+1.70 log-points), and scale absorbs ~78% of it (+1.70 → +0.38) - the "budget illusion" again.

2. What remains after both adjustments is +0.34 (≈ +40% ROI, p = 0.0006), and every scale-adjusted CI stays well above zero - unlike H1, the effect does not disappear.

3. The raw-ROI comparison at the end of this notebook agrees (full model +0.67, p = 0.0003): the franchise verdict holds on both scales.

#### *Verdict : Franchise premium survives*

### 2. Remakes

- What does the unadjusted comparison say? (Remake)

In [11]:
X = df['Remake']
X = sm.add_constant(X)
rawmodel_r = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel_r.params['Remake'], rawmodel_r.pvalues['Remake']
lo, hi = rawmodel_r.conf_int().loc['Remake']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +1.6008  
p-value = 0.0000   
95% CI [+1.4332, +1.7684]


- What does a content-adjusted (scale not yet controlled) model say? (Remake)

In [12]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel_r = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel_r.params['Remake'], ROImodel_r.pvalues['Remake']
lo, hi = ROImodel_r.conf_int().loc['Remake']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +1.0892  
p-value = 0.0000   
95% CI [+0.9064, +1.2721]


- What does a scale adjustment alone model say? (Remake)

In [13]:
X = "log_ROI ~ Remake + C(Scale_Bin)"
raw_scale_r = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale_r.params['Remake[T.True]'], raw_scale_r.pvalues['Remake[T.True]']
lo, hi = raw_scale_r.conf_int().loc['Remake[T.True]']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +0.3397  
p-value = 0.0002   
95% CI [+0.1602, +0.5191]


- What does a full model considering everything say? (Remake)

In [14]:
X = "log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale_r = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale_r.params['Remake[T.True]'], full_scale_r.pvalues['Remake[T.True]']
lo, hi = full_scale_r.conf_int().loc['Remake[T.True]']
print(f"Remake coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Remake coefficient : +0.3488  
p-value = 0.0001   
95% CI [+0.1688, +0.5288]


### Result: **Remake premium survives — more clearly than on raw ROI** (response = log(ROI))

| Model | Remake effect, log(ROI) | ≈ % ROI | 95% CI | p-value |
|---|---|---|---|---|
| Unadjusted | *+1.6008* | +396% | [+1.4332, +1.7684] | < 0.0001 |
| Content-adjusted, scale not controlled | *+1.0892* | +197% | [+0.9064, +1.2721] | < 0.0001 |
| Scale-adjusted only | *+0.3397* | +40% | [+0.1602, +0.5191] | 0.0002 |
| Scale + content (full) | *+0.3488* | +42% | [+0.1688, +0.5288] | 0.0001 |


1. Scale absorbs ~79% of the unadjusted remake premium (+1.60 → +0.34).

2. Unlike on raw ROI - where the scale-only model was borderline (p = 0.079) - on the validated log scale the remake premium is significant at every adjusted stage (p ≤ 0.0002). The borderline case was noise from the raw scale's error funnel, not a weak effect.

3. In the full model the premium is +0.35 (≈ +42% ROI, p = 0.0001), on par with franchise's +40% - remakes and franchises earn comparable like-for-like premiums, and the raw-ROI comparison at the end confirms both.

#### *Verdict : Remake premium survives*

## H3 : Genre composition at comparable scale
**Do some genres out-earn mainstream films once scale is held fixed?**

### The check: all 13 genres, 4-stage test (raw → content → scale → full)

- For each genre: the same 4-stage ladder as H1/H2; reference category = action films
  (the first genre, dropped during one-hot encoding to avoid the dummy variable trap);
  all p-values are unadjusted, compared against α = 0.05

In [15]:
genres = [c for c in focus_cols if c.startswith('Genre_')]
content_model = sm.OLS(Y, sm.add_constant(df[focus_cols])).fit(cov_type='HC3')
full_model    = smf.ols("log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)", data=df).fit(cov_type='HC3')

def gterm(model, g):
    k = next(k for k in model.params.index if k.split('[')[0] == g)
    return model.params[k], model.pvalues[k]

rows = []
for g in genres:
    m_raw   = sm.OLS(Y, sm.add_constant(df[g])).fit(cov_type='HC3')
    m_scale = smf.ols(f"log_ROI ~ {g} + C(Scale_Bin)", data=df).fit(cov_type='HC3')
    rc, rp = gterm(m_raw, g);  cc, cp = gterm(content_model, g)
    sc, sp = gterm(m_scale, g); fc, fp_ = gterm(full_model, g)
    rows.append((g.replace('Genre_',''), int(df[g].sum()),
                 f"{rc:+.3f} (p={rp:.3f})", f"{cc:+.3f} (p={cp:.3f})",
                 f"{sc:+.3f} (p={sp:.3f})", f"{fc:+.3f} (p={fp_:.3f})"))

stage = pd.DataFrame(rows, columns=['genre','n','raw','content','scale','full'])
stage = stage.sort_values('full', ascending=False)
print(stage.to_string(index=False))

       genre   n              raw          content            scale             full
     fantasy  13 -0.916 (p=0.061) -0.871 (p=0.026) -1.216 (p=0.000) -1.086 (p=0.000)
   animation   5 -1.127 (p=0.108) -1.777 (p=0.071) -0.700 (p=0.330) -0.892 (p=0.228)
    suspense  29 -1.058 (p=0.000) -0.769 (p=0.006) -0.347 (p=0.145) -0.212 (p=0.398)
  love_story 131 -0.185 (p=0.258) -0.204 (p=0.248) -0.258 (p=0.018) -0.144 (p=0.294)
      comedy 282 +0.298 (p=0.004) -0.050 (p=0.701) -0.023 (p=0.755) -0.001 (p=0.993)
       adult  76 +0.553 (p=0.000) +0.622 (p=0.000) +1.763 (p=0.000) +1.624 (p=0.000)
 documentary   1 +1.743 (p=0.000) +2.582 (p=0.675) +0.236 (p=0.990) +0.824 (p=0.837)
      horror  54 +0.441 (p=0.016) +0.199 (p=0.318) +0.402 (p=0.005) +0.438 (p=0.013)
     rom_com  97 +0.652 (p=0.000) +0.444 (p=0.016) +0.119 (p=0.361) +0.189 (p=0.217)
      masala  12 +1.918 (p=0.000) +0.858 (p=0.002) +0.200 (p=0.246) +0.188 (p=0.329)
mythological  14 -1.275 (p=0.000) -0.715 (p=0.019) -0.351 (p=0.35

**Result :** 
- At the full stage (scale + content, log(ROI)), two genres keep a significant positive premium: **adult** (+1.624, ≈ +407% ROI, p < 0.0001) and **horror** (+0.438, ≈ +55% ROI, p = 0.013), while **fantasy** is confirmed as a significant negative (−1.086, ≈ −66% ROI, p < 0.0001).
- The premiums that looked significant on raw ROI — rom_com (p = 0.217), drama (p = 0.803) and the one-film documentary (p = 0.837) — dissolve on the validated log scale; they were artifacts of the raw scale's error funnel, not genre effects. Drama's story is the cleanest example: significantly *negative* unadjusted (−0.614, p < 0.001), exactly null at the full stage (+0.026, p = 0.803).
- Since adult retains significance through every stage, **H3 is not rejected — but it is scoped to the adult genre** (with horror as a secondary, smaller effect).
- The genre with by far the largest premium, adult, is examined in depth below.

### Full Scale Check for Adult films

- What does the unadjusted comparison say?

In [16]:
X = df['Genre_adult']
X = sm.add_constant(X)
rawmodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = rawmodel.params['Genre_adult'], rawmodel.pvalues['Genre_adult']
lo, hi = rawmodel.conf_int().loc['Genre_adult']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : +0.5528  
p-value = 0.0000   
95% CI [+0.3337, +0.7719]


- What does a content-adjusted (scale not yet controlled) model say?

In [17]:
X = df[focus_cols]
X = sm.add_constant(X)
ROImodel = sm.OLS(Y, X).fit(cov_type='HC3')
c, p = ROImodel.params['Genre_adult'], ROImodel.pvalues['Genre_adult']
lo, hi = ROImodel.conf_int().loc['Genre_adult']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : +0.6224  
p-value = 0.0000   
95% CI [+0.3251, +0.9197]


- What does a scale adjustment alone model say?

In [18]:
X = "log_ROI ~ Genre_adult + C(Scale_Bin)"
raw_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = raw_scale.params['Genre_adult[T.True]'], raw_scale.pvalues['Genre_adult[T.True]']
lo, hi = raw_scale.conf_int().loc['Genre_adult[T.True]']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : +1.7626  
p-value = 0.0000   
95% CI [+1.4524, +2.0728]


- What does a full model considering everything say?

In [19]:
X = "log_ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)"
full_scale = smf.ols(X, data=df).fit(cov_type='HC3')
c, p = full_scale.params['Genre_adult[T.True]'], full_scale.pvalues['Genre_adult[T.True]']
lo, hi = full_scale.conf_int().loc['Genre_adult[T.True]']
print(f"Genre_adult coefficient : {c:+.4f}  \np-value = {p:.4f}   \n95% CI [{lo:+.4f}, {hi:+.4f}]")

Genre_adult coefficient : +1.6236  
p-value = 0.0000   
95% CI [+1.2624, +1.9847]


#### Result: **H3 is supported — and the log scale tells it better**

| Model | Adult effect, log(ROI) | ≈ % ROI | 95% CI | p-value |
|---|---|---|---|---|
| Unadjusted | *+0.5528* | +74% | [+0.3337, +0.7719] | < 0.0001 |
| Content-adjusted, scale not controlled | *+0.6224* | +86% | [+0.3251, +0.9197] | < 0.0001 |
| Scale-adjusted only | *+1.7626* | +483% | [+1.4524, +2.0728] | < 0.0001 |
| Scale + content (full) | *+1.6236* | +407% | [+1.2624, +1.9847] | < 0.0001 |


1. On the geometric (log) view adult films out-earn comparable films at every stage - even unadjusted (+0.55). The raw-ROI model had shown a *negative* unadjusted gap (−0.13): adult films rarely produce tail hits, and the arithmetic mean punishes them for it, while the log (geometric) mean rewards their consistency. Both facts are real; the log scale is the one the distribution fitting validated.

2. Once films are compared within their own scale band, the premium grows to +1.76 log-points and settles at +1.62 (≈ +407% ROI, p < 0.0001) in the full model - the strongest and most precisely estimated effect in the project.

3. The claim stays scoped: adult films exist only in the small scale bands (76 films, none above scale 5), and the reference category is action films (dropped during one-hot encoding to avoid the dummy variable trap).

#### *Verdict : At comparable scale, the adult genre out-earns mainstream - on the validated log scale the premium is large, positive, and visible at every stage*

## H4 : Star Power
**Is revenue concentrated among a few recurring stars and directors?**

### 1. Concentration: Gini and top-10 share

**What Gini is:** line up all names poorest → richest; Gini (0 to 1) measures how far that line bends away from perfectly equal revenue — 0 = everyone identical, 1 = one name holds everything.

**Why it is necessary:** the top-10 share depends on an arbitrary cutoff (top-5? top-20?); Gini summarises the whole distribution in one cutoff-free number.

**Here:** names are stars/directors with revenue summed across their films, so "recurring" becomes visible.

- Revenue is summed per name first — concentration is a name-level measure (H4 asks about names, not films)

In [20]:
def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return 2*np.sum(np.arange(1, n+1)*x)/(n*x.sum()) - (n+1)/n

def concentration(names, values):
    tot = pd.DataFrame({'name': names, 'val': values}).groupby('name')['val'].sum()
    return gini(tot.values), tot.nlargest(10).sum()/tot.sum(), tot

star_gini, star_top10, star_tot = concentration(df['Lead Star'], df['Revenue'])
dir_gini, dir_top10, dir_tot = concentration(df['Director'], df['Revenue'])

print(f"Stars    : {len(star_tot)} names | Gini = {star_gini:.3f} | top-10 share = {100*star_top10:.1f}%")
print(f"Directors: {len(dir_tot)} names | Gini = {dir_gini:.3f} | top-10 share = {100*dir_top10:.1f}%")
print("\nTop-10 stars by total revenue:")
for nm, v in star_tot.nlargest(10).items():
    print(f"  {nm:25s} {v/1e9:6.2f} B  ({100*v/star_tot.sum():.1f}%)")

Stars    : 761 names | Gini = 0.954 | top-10 share = 59.3%
Directors: 1045 names | Gini = 0.897 | top-10 share = 25.6%

Top-10 stars by total revenue:
  Salman Khan                46.13 B  (11.4%)
  Akshay Kumar               41.05 B  (10.2%)
  Shahrukh Khan              31.90 B  (7.9%)
  Aamir Khan                 29.03 B  (7.2%)
  Ajay Devgn                 24.39 B  (6.0%)
  Ranbir Kapoor              17.22 B  (4.3%)
  Hrithik Roshan             16.95 B  (4.2%)
  Saif Ali Khan              12.05 B  (3.0%)
  Ranveer Singh              11.08 B  (2.7%)
  Prabhas                     9.70 B  (2.4%)


### 2. Scale-managed star ranking: does star identity matter beyond scale?

- Gini tells us the revenue is concentrated by the top 10%, but it never tells us if this is because of the movies themselves or the stars.
> - If the films themselves are very unequal (a few blockbusters hold most of the revenue), then whoever is tagged to those films *looks* like a star. A high Gini can therefore be false concentration: scale, not stardom.
> - We seek to remove such cases here
- **ROI ~ C(star) + C(Scale_Bin):** manage scale first, then rank stars by their scale-adjusted premium; stars with at least 5 films only (a 1-film star ranks by pure residual luck)

In [21]:
df['star'] = df['Lead Star']
m_scale = smf.ols("log_ROI ~ C(Scale_Bin)", data=df).fit()
m_star  = smf.ols("log_ROI ~ C(star) + C(Scale_Bin)", data=df).fit()
print(f"R2 with scale bands only : {m_scale.rsquared:.3f}")
print(f"R2 with scale + star name: {m_star.rsquared:.3f}   (stars add {m_star.rsquared-m_scale.rsquared:+.3f})")

vc = df['star'].value_counts()
rows = []
for k in m_star.params.index:
    if k.startswith('C(star)'):
        star = k.split('[')[1].rstrip(']').strip("'T.")
        rows.append((star, int(vc.get(star, 0)), m_star.params[k]))
rank = pd.DataFrame(rows, columns=['star', 'n_films', 'premium'])
rank = rank[rank.n_films >= 5].sort_values('premium', ascending=False)
print(f"\nTop 10 stars by scale-adjusted premium (>=5 films required, {len(rank)} eligible):")
print(rank.head(10).round(3).to_string(index=False))

R2 with scale bands only : 0.488
R2 with scale + star name: 0.804   (stars add +0.316)



Top 10 stars by scale-adjusted premium (>=5 films required, 72 eligible):
         star  n_films  premium
        Sapna        9    5.981
   Aamir Khan       10    5.021
 Varun Dhawan        7    4.724
Shahrukh Khan       16    4.639
  Vidya Balan        7    4.553
  Irrfan Khan        9    4.461
   Imran Khan        9    4.446
 Shiney Ahuja        7    4.421
Ranveer Singh        9    4.416
 Arjun Kapoor        6    4.388


#### Result: **H4 is supported**

| | Stars | Directors |
|---|---|---|
| Names | 761 | 1045 |
| Gini | *0.954* | *0.897* |
| Top-10 share (observed) | *59.3%* | *25.6%* |
| Top-10 share under perfect equality | 1.3% | 1.0% |


1. The concentration itself is extreme: the top-10 stars hold 59.3% of all revenue against just 1.3% under perfect equality (10/761), and the Gini of 0.954 confirms near-monopoly.

2. The scale-managed check (on log(ROI)): explaining log(ROI) with scale bands alone gives R2 = 0.488; adding star names lifts it to 0.804 - star identity adds +0.32 of explained variance beyond film size, even more than the raw-ROI models suggested.

3. The scale-adjusted ranking (stars with at least 5 films, log(ROI) premiums) puts Sapna (+5.98, 9 films), Aamir Khan (+5.02) and Varun Dhawan (+4.72) at the top. Premiums are log-points, so they serve as ranking statistics rather than percentage claims: within low scale bands the baseline ROI is small, which inflates ratios.

4. Caveats: the ranking uses point estimates only (a 760-dummy model has no valid HC3 errors), a single mega-hit or a consistent small-film record can lift a star's log premium, directors are far less concentrated than stars (25.6% vs 59.3%), and this is association, not causation - big films make stars as much as stars make big films.

#### *Verdict : Revenue concentration is real and extreme, and star identity keeps explaining returns even after film scale is accounted for*

## Scale-robustness check: the same ladders on raw ROI
Every verdict above was drawn on **log(ROI)** — the scale 3.DistributionFitting.ipynb validated and
4.LinearRegression.ipynb adopted. To show the conclusions are not an artifact of that choice, the same
4-stage ladders (unadjusted → content → scale → full, HC3) are re-fit here on **raw ROI**. Coefficients
are absolute ROI-point effects again; compare with the log ladders above.

In [22]:
def ladder4_raw(flag):
    """unadjusted -> content -> scale -> full ladder for one flag, response = raw ROI, HC3."""
    m_raw   = sm.OLS(df['ROI'], sm.add_constant(df[flag].astype(int))).fit(cov_type='HC3')
    m_cont  = sm.OLS(df['ROI'], sm.add_constant(df[focus_cols])).fit(cov_type='HC3')
    m_scale = smf.ols(f"ROI ~ {flag} + C(Scale_Bin)", data=df).fit(cov_type='HC3')
    m_full  = smf.ols("ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)", data=df).fit(cov_type='HC3')
    for stage, m in [('unadjusted', m_raw), ('content', m_cont), ('scale', m_scale), ('full', m_full)]:
        k = next(k for k in m.params.index if k.split('[')[0] == flag)
        print(f"  {stage:10s} {m.params[k]:+.4f}  (p = {m.pvalues[k]:.4f})")

raw_full = smf.ols("ROI ~ " + " + ".join(focus_cols) + " + C(Scale_Bin)", data=df).fit(cov_type='HC3')
rows = []
for g in [c for c in focus_cols if c.startswith('Genre_')]:
    k = next(k for k in raw_full.params.index if k.split('[')[0] == g)
    rows.append((g.replace('Genre_', ''), int(df[g].sum()), raw_full.params[k], raw_full.pvalues[k]))
genre_raw = pd.DataFrame(rows, columns=['genre', 'n', 'coef', 'p']).sort_values('coef', ascending=False)

for flag in ['Holiday', 'Franchise', 'Remake']:
    print(f"=== {flag} on raw ROI (HC3) ===")
    ladder4_raw(flag)
    print()
print("=== Genres on raw ROI, full stage (HC3) ===")
print(genre_raw.round(4).to_string(index=False))

=== Holiday on raw ROI (HC3) ===
  unadjusted +0.2262  (p = 0.0000)
  content    +0.1351  (p = 0.0061)
  scale      +0.0671  (p = 0.1388)
  full       +0.0598  (p = 0.1802)

=== Franchise on raw ROI (HC3) ===


  unadjusted +1.3794  (p = 0.0000)
  content    +1.1665  (p = 0.0000)
  scale      +0.6183  (p = 0.0005)
  full       +0.6676  (p = 0.0003)

=== Remake on raw ROI (HC3) ===


  unadjusted +0.9555  (p = 0.0000)
  content    +0.7050  (p = 0.0000)
  scale      +0.2380  (p = 0.0785)
  full       +0.3012  (p = 0.0222)

=== Genres on raw ROI, full stage (HC3) ===
       genre   n    coef      p
       adult  76  0.5446 0.0000
 documentary   1  0.4776 0.0422
      masala  12  0.3817 0.2386
     rom_com  97  0.3559 0.0067
  love_story 131  0.1767 0.1250
      comedy 282  0.1737 0.0746
       drama 642  0.1640 0.0497
      horror  54  0.1614 0.2028
mythological  14  0.1460 0.1770
    thriller 216  0.0647 0.4862
    suspense  29  0.0014 0.9889
     fantasy  13 -0.4356 0.0009
   animation   5 -0.4793 0.1105


**Result — the verdicts do not depend on the scale choice, and where they differ, the log scale is the trustworthy one.**

| Verdict | log(ROI) — primary | raw ROI — labelled comparison |
|---|---|---|
| H1 Holiday | full −0.02 (p = 0.79) — rejected | full +0.06 (p = 0.18) — rejected |
| H2 Franchise | full +0.34 (p = 0.0006) — survives | full +0.67 (p = 0.0003) — survives |
| H2 Remake | full +0.35 (p = 0.0001) — survives | full +0.30 (p = 0.022) — survives |
| H3 genres | adult +1.62 (p < 0.0001), horror +0.44 (p = 0.013) | adult +0.54 (p < 0.0001), rom_com +0.36 (p = 0.007), drama +0.16 (p = 0.0497), documentary +0.48 (p = 0.042) |

1. H1 and H2 agree on both scales: holiday never survives, franchise and remake always do. The remake scale-only borderline case on raw ROI (p = 0.079) resolves clearly on the validated log scale (p = 0.0002).

2. H3 is where the scales disagree: rom_com, drama and the one-film documentary are significant only on raw ROI. 3.DistributionFitting.ipynb validated log(ROI) as the near-normal scale (raw-ROI skew +2.23 vs log −0.36), and 4.LinearRegression.ipynb showed the raw scale's error funnel manufactures exactly such fragile significances - so H3 is scoped to adult, with horror emerging on the log scale.

3. Coefficients are not comparable across columns by design: log coefficients are percentage effects (exp(β) − 1), raw coefficients are absolute ROI points.